In [1]:
import torch
from diffusion.approaches.matching.alphas_betas import LinearAlpha, LinearBeta
from diffusion.approaches.matching.prob_paths import GaussianCondProbPath
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.data.mnist_sampler import MNISTSampleable
from diffusion.architectures.res_unet import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable()

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=(1, 32, 32),
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
)

trainer = FlowTrainer(
    path=path,
    model=backbone,
    num_classes=sampeable.num_classes,
)

In [ ]:
trainer.train(
    num_epochs=5,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=False,
)

In [5]:
torch.save(backbone.state_dict(), "backbone.pt")